# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulwasay45/flyrankinternship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Five plain-words contract answers for my lane (Refresh / Content Opportunity Scoring):**

1. **One row means:** one content item (page) on one day — grain of `fact_content_daily_performance` is `report_date × client_hash_id × content_hash_id`. For modeling I will aggregate to **one row = one page** over a chosen window.

2. **Tables I will use:**  
   - `fact_content_daily_performance` (daily signals)  
   - `dim_content` (content metadata)  
   - optionally `dim_clients` for history coverage

3. **Time window:** I develop on a mid-panel month (`month=2026-03`) so I never touch the final month (June 2026), which is reserved as a sealed test / outcome window.

4. **What I predict / rank:** a priority score for “should an editor review this page for refresh?”  
   Proxy label: did impressions drop >20% from the previous 30-day window to the next 30-day window (observed outcome).

5. **One thing I deliberately exclude:**  
   - Any product decision flag / health score / priority score (they are not in the release and would leak).  
   - `trend_pct` or any future-window metric used as a feature.  
   - Raw client names, domains, URLs, titles, or private queries.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Field classification (starter + warehouse view):**

| Bucket | Fields | Why |
|--------|--------|-----|
| **Feature** | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, days since last update / content age (from dim), word_count, content_type, engagement/scroll signals when available | Knowable at the decision moment (before the outcome window) |
| **Label / proxy** | Future decline flag built from later-window impressions (e.g. last-30 vs prev-30) | Observed outcome — never a feature |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date` | Grouping, joining, client-holdout splits only |
| **Excluded** | Product health/priority scores, action flags, raw URLs/titles/queries, any metric from the future window | Leakage or private / not in release |

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# Setup: DuckDB + Hugging Face (token from Colab Secret only)
# ============================================================
%pip -q install duckdb huggingface_hub

import os, getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

# Mid-panel month only — never the final month for label development
MONTH = "2026-03"
FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

print("Connected. Working on month =", MONTH)

# ------------------------------------------------------------
# QUERY 1 — Grain check
# One row should be unique on (report_date, client_hash_id, content_hash_id)
# ------------------------------------------------------------
print("\n=== QUERY 1: Grain check ===")
grain = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MONTH}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print("Rows that violate the grain (should be empty):", len(grain))
print(grain)

# ------------------------------------------------------------
# QUERY 2 — Row count + date span of my slice
# ------------------------------------------------------------
print("\n=== QUERY 2: Counts and date span ===")
stats = con.sql(f"""
    SELECT
        COUNT(*)                    AS n_rows,
        COUNT(DISTINCT content_hash_id) AS n_pages,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        MIN(report_date)            AS min_date,
        MAX(report_date)            AS max_date
    FROM {FACT_MONTH}
""").df()
print(stats.to_string(index=False))

# ------------------------------------------------------------
# QUERY 3 — Availability (filter with IS TRUE)
# Show how many rows survive a real availability filter
# ------------------------------------------------------------
print("\n=== QUERY 3: Availability (IS TRUE) ===")
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_true,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_true,
        SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS both_true
    FROM {FACT_MONTH}
""").df()
print(avail.to_string(index=False))
print("Note: always filter with IS TRUE / IS NOT TRUE — never = FALSE alone.")

# ------------------------------------------------------------
# FIVE FEATURES — small feature frame for my lane
# Aggregated to one row = one page for the chosen month
# ------------------------------------------------------------
print("\n=== FIVE FEATURES (one row = one page) ===")
features = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions)                         AS impressions_month,
        SUM(gsc_clicks)                              AS clicks_month,
        AVG(gsc_avg_position)                        AS avg_position_month,
        SUM(gsc_clicks) * 100.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr_month,
        COUNT(DISTINCT report_date)                  AS days_with_data
    FROM {FACT_MONTH}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 50
""").df()

print(f"Feature frame shape: {features.shape}")
print(features.head(6))

print("""
Feature availability lines (knowable at decision moment because…):
1. impressions_month  — known from past search data already collected
2. clicks_month       — known from past search data already collected
3. avg_position_month — known from past GSC position already collected
4. ctr_month          — derived only from past impressions + clicks
5. days_with_data     — count of past days that had any data
""")

# ------------------------------------------------------------
# THE TRAP — deliberate leakage, then remove it
# ------------------------------------------------------------
print("\n=== THE TRAP (deliberate leak → then remove) ===")

# Fake a simple label from the same window (this is the leak)
features["leaky_label"] = (features["impressions_month"] < features["impressions_month"].median()).astype(int)

# Add a label-derived column on purpose
features["leaky_feature"] = features["leaky_label"]   # THIS IS THE TRAP

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

X_leaky = features[["impressions_month", "clicks_month", "avg_position_month", "ctr_month", "days_with_data", "leaky_feature"]].fillna(0)
y = features["leaky_label"]

Xtr, Xte, ytr, yte = train_test_split(X_leaky, y, test_size=0.3, random_state=42, stratify=y)
tree_leaky = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xtr, ytr)
print(f"WITH leaky feature  → accuracy: {tree_leaky.score(Xte, yte):.3f}   ← looks almost perfect (trap)")

# Now delete the leaky feature and keep the honest number
X_honest = features[["impressions_month", "clicks_month", "avg_position_month", "ctr_month", "days_with_data"]].fillna(0)
Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
tree_honest = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xtr, ytr)
print(f"WITHOUT leaky feature → accuracy: {tree_honest.score(Xte, yte):.3f}   ← honest number we keep")

print("\nLesson: the jump to near-perfect score was pure leakage. We delete the column and keep the honest result.")

Connected. Working on month = 2026-03

=== QUERY 1: Grain check ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows that violate the grain (should be empty): 0
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []

=== QUERY 2: Counts and date span ===
 n_rows  n_pages  n_clients   min_date   max_date
9841378   331437         55 2026-03-01 2026-03-31

=== QUERY 3: Availability (IS TRUE) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  gsc_true  ga4_true  both_true
    9841378 3611061.0  413966.0   364347.0
Note: always filter with IS TRUE / IS NOT TRUE — never = FALSE alone.

=== FIVE FEATURES (one row = one page) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (116114, 7)
            content_hash_id           client_hash_id  impressions_month  \
0  content_39d7361b4945d504  client_62f4a7e64f5e0096               77.0   
1  content_d49a012dcb924e31  client_62f4a7e64f5e0096              329.0   
2  content_614baf2af4330bd7  client_62f4a7e64f5e0096              772.0   
3  content_225dc9235023be5f  client_62f4a7e64f5e0096              488.0   
4  content_7dbc094b799e05a4  client_62f4a7e64f5e0096              705.0   
5  content_40b10da45f4c1cb5  client_62f4a7e64f5e0096               50.0   

   clicks_month  avg_position_month  ctr_month  days_with_data  
0           0.0            4.074107   0.000000              24  
1           0.0            5.177774   0.000000              31  
2           1.0            4.685335   0.129534              31  
3           1.0           17.148172   0.204918              31  
4           1.0            5.956862   0.141844              31  
5           0.0           12.977513   0.000000     

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation of my slice:**

1. **Unbalanced panel** — different clients have very different history lengths. I must check `dim_clients.gsc_data_start` / `ga4_data_start` before defining any time window; otherwise “no data yet” is mistaken for “zero traffic.”

2. **GSC-only early rows** — before a client’s GA4 start date the GA4 columns are zero-filled or NULL. Always filter with `ga4_data_available IS TRUE` (never `= FALSE` alone).

3. **I develop only on mid-panel months** (here `2026-03`). The final month (June 2026) is sealed as a future test / outcome window so I never train on the period I will later try to predict.

4. **This contract cannot prove causation.** A high priority score only means “this page looks like pages that historically declined.” It does not prove that a refresh will recover traffic.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.